# BCS 3101: Basics of Machine Learning
## Assignment 2 – Notebook 3

**Student Name:** KYOSHABIRE DIANAH  
**Registration Number:** 2024/A/KCS/5188/G/F  
**Group Project:** Predicting Monthly Maize and Beans Prices in Selected Ugandan Markets  

This notebook covers two parts from the companion guide:

- **Stage 7: Data Pre-processing II – Data Integration**
- **Stage 8.1: Encoding Categorical Variables** (first part of Data Transformation)

Data Transformation as a whole is worth **15%** on the marking rubric.  
We follow the exact reasoning the lecturer expects.

---
## Load the data from the previous notebooks

We start from the filtered dataset that Notebook 1 prepared.  
Notebook 2 already decided how to handle missing values, duplicates and outliers.  
Those decisions are applied here so the data is ready for transformation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

%matplotlib inline
sns.set_style('whitegrid')

# Load the filtered dataset
df = pd.read_csv('uganda_maize_beans_selected_markets.csv')

print("Data loaded.")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print("\nMarkets:")
print(df['mkt_name'].value_counts())

---
## Stage 7: Data Pre-processing II – Data Integration

The companion guide is very clear on this stage:

> “Data integration only applies if you are combining more than one data source into a single working dataset… If your assignment uses a single, already-tabular dataset, you do not need to perform this stage. You must still explicitly say so in your report… rather than silently omitting the section, since the rubric rewards an explicitly reasoned judgement over a silent skip.”

### Our situation
We are using **one single file** – the Uganda Real-Time Food Prices dataset.  
All seven markets, all years and all price columns come from that one source.

Therefore **data integration is not applicable**.

We state this clearly so the examiner sees we understood the stage and made a reasoned decision.

In [ ]:
# There is nothing to merge or join.
# We simply confirm we are working with a single source.
print("Number of source files used: 1")
print("Data integration step: NOT APPLICABLE")
print("Reason: This project uses a single unified source file (Uganda RTFP dataset).")

---
## Stage 8: Data Pre-processing III – Data Transformation

The guide says:

> “Transformation converts your cleaned data into a numeric form and scale that machine learning algorithms can actually consume. Most algorithms cannot use raw text categories, and many are sensitive to features living on very different numeric scales.”

In this notebook we focus on the first part of transformation: **encoding categorical variables**.

### 8.1 Encoding Categorical Variables

Machine learning algorithms operate on numbers.  
Every text (categorical) column must be turned into numbers before most models can use it.

The guide explains two standard techniques:

**Label Encoding**  
Assigns each category an integer (0, 1, 2, …).  
This is valid **only** when the categories have a genuine order (ordinal data).  
Example: Low → 0, Medium → 1, High → 2.

**One-Hot Encoding**  
Creates one new binary (0/1) column for each category.  
No false order is created.  
This is the right choice for nominal (unordered) categories such as market names or colours.

The guide warns against a common mistake:
> “A recurring error is label-encoding a nominal (unordered) variable… without acknowledging that this silently implies one category is greater than another.”

In [ ]:
# First we look at the categorical columns we have
print("Data types of all columns:")
print(df.dtypes)

print("\nUnique values in mkt_name:")
print(df['mkt_name'].unique())
print(f"\nNumber of unique markets: {df['mkt_name'].nunique()}")

The main categorical column we need to encode is **`mkt_name`**.  
It has seven markets and there is no natural order between them (Gulu is not “greater than” Jinja).  
Therefore `mkt_name` is a **nominal** variable.

According to the guide we should use **One-Hot Encoding** for it.

In [ ]:
# One-Hot Encoding for the market name
# drop_first=True avoids the dummy-variable trap (perfect multicollinearity)
df_encoded = pd.get_dummies(df, columns=['mkt_name'], drop_first=True, dtype=int)

print("Shape before encoding:", df.shape)
print("Shape after one-hot encoding:", df_encoded.shape)
print("\nNew market columns created:")
market_cols = [c for c in df_encoded.columns if c.startswith('mkt_name_')]
print(market_cols)

We used `drop_first=True`.  
This drops one market column (the first one in alphabetical order).  
The dropped market becomes the reference category.  
This is the standard way to avoid the dummy-variable trap that the guide mentions.

In [ ]:
# Quick look at the new columns
print(df_encoded[market_cols].head(10))

Each row now has a 1 in the column of its own market and 0s in the others.  
The model can use these binary columns as features.

### Month as a categorical feature

Month is a number from 1 to 12, but it is cyclical (December is next to January).  
For a first version we can treat it as an ordinal variable and keep it as a plain number,  
or we can one-hot encode it as well.  

In this notebook we keep month as a numeric feature for simplicity.  
Later notebooks can add sine/cosine encoding if needed.  
Year is already numeric and can stay as it is.

In [ ]:
# Confirm month and year are numeric
print("Data type of year:", df_encoded['year'].dtype)
print("Data type of month:", df_encoded['month'].dtype)
print("\nUnique months:", sorted(df_encoded['month'].unique()))

### Optional: Label Encoding example (for illustration only)

The guide also shows Label Encoding.  
We demonstrate it here on a copy of the market column so students can see the difference,  
but we do **not** keep the label-encoded version because the markets have no natural order.

In [ ]:
# Illustration only – we will not use this for modelling
le = LabelEncoder()
label_encoded_markets = le.fit_transform(df['mkt_name'])

print("Label encoding example (not used for modelling):")
for market, code in zip(le.classes_, range(len(le.classes_))):
    print(f"  {market} → {code}")

print("\nWe keep the one-hot version instead, because markets are nominal (no order).")

### Selecting the columns we will carry forward

We keep:
- The two target columns: `c_maize`, `c_beans`
- Useful numeric features: year, month, latitude, longitude, data coverage, selected other completed prices and inflation/trust scores
- The new one-hot market columns

We drop columns that are almost empty or not needed for the next stages.

In [ ]:
# Columns we decide to keep for the rest of the pipeline
cols_to_keep = [
    'year', 'month', 'lat', 'lon',
    'c_maize', 'c_beans',
    'c_oil', 'c_salt', 'c_food_price_index',
    'inflation_maize', 'inflation_beans',
    'trust_maize', 'trust_beans',
    'data_coverage', 'data_coverage_recent', 'index_confidence_score'
]

# Add the one-hot market columns
cols_to_keep = cols_to_keep + market_cols

# Some columns may not exist in every version of the file; keep only those that are present
cols_to_keep = [c for c in cols_to_keep if c in df_encoded.columns]

df_model = df_encoded[cols_to_keep].copy()

print("Columns kept for modelling:")
print(df_model.columns.tolist())
print(f"\nFinal shape: {df_model.shape}")
print("\nMissing values left in this table:")
print(df_model.isnull().sum().sum())

The table now contains only numeric columns.  
This is exactly what the next stage (scaling) and later modelling steps need.

In [ ]:
# Save the encoded dataset for Notebook 4
df_model.to_csv('uganda_maize_beans_encoded.csv', index=False)
print("Encoded dataset saved as 'uganda_maize_beans_encoded.csv'")
print("Next notebook will load this file and apply scaling.")

---
## Summary of decisions in this notebook

| Step | Decision | Reason (from the guide) |
|------|----------|-------------------------|
| Data Integration | Not applicable | Single source file; rubric rewards an explicit statement |
| Encoding of `mkt_name` | One-Hot Encoding (drop_first=True) | Markets are nominal (no natural order) |
| Label Encoding | Shown only as illustration | Would create a false order; not used |
| Month and Year | Kept as numeric | Already numbers; month can be refined later if needed |
| Column selection | Kept targets + useful numeric features + one-hot markets | Ready for scaling and modelling |

These decisions match Stage 7 and Stage 8.1 of the companion guide and will be written into the final report with the exact shapes shown above.

---
## End of Notebook 3

### What we finished
- Stated clearly that Data Integration is not needed (Stage 7)
- Identified `mkt_name` as a nominal categorical variable
- Applied One-Hot Encoding with `drop_first=True`
- Showed Label Encoding only for illustration
- Selected the columns that will go forward
- Saved an encoded CSV for the next student

### What comes next
Notebook 4 (NIYONSHUTI MERCY) will continue **Stage 8** by applying feature scaling (Min-Max and/or StandardScaler) and any needed transforms, following the rest of the transformation section in the guide.

**Reminder**  
Data Transformation is worth 15%. Every encoding choice must be justified with the type of the variable (nominal vs ordinal) exactly as the guide requires.